# Practical 02a — Symbolic Manipulation and Pattern Matching (Step-by-Step Edition)

**Artificial Intelligence 3(2-1) · BS Computer Science · Semester III**

| | |
|---|---|
| **Week** | 2 |
| **Related lectures** | Lectures 4, 18 and 19 |
| **Duration** | 3 hours |
| **Covers the same work as** | Practical 02 (same concepts, same five tasks, same marks) |
| **Submission** | This notebook, run top to bottom from a clean kernel |

This is an easier route through Practical 02. Nothing is left out — you still build a
pattern matcher, a chatbot-style responder and an expression simplifier — but every idea
is introduced in a small step, most of the code is written for you, and you fill in the
missing pieces. A **CHECK** cell after each exercise tells you whether you got it right.

## 0. How to use this notebook

1. **Run cells in order**, top to bottom. Click a cell and press **Shift + Enter**.
2. When you see **`___`** (three underscores) in code, that is a **blank** for you to fill in.
   Replace the `___` with your answer, then run the cell again.
3. Run the **CHECK** cell that follows. It prints:
   - `OK` — correct, move on.
   - `NOT YET` — it shows what it expected, what you gave, and a hint.
4. If you see a red error message, the most common cause is a blank you have not filled yet.
   Read the **last line** of the error — it names the problem.
5. If you restart the notebook (Kernel → Restart), run all the cells above the one you are
   on again, starting from the setup cell.

> 💡 **Predict before you run.** Many cells ask *"What do you think this will print?"*
> Say your guess out loud (or write it in a comment) before pressing Shift + Enter. This is
> the fastest way to learn.

## 1. Objectives

By the end of this session you will be able to:

1. Represent statements as structured data (a list of words) rather than as plain strings.
2. Write a pattern matcher that recognises a template containing variables.
3. Build a transformation system that rewrites one expression into another.
4. Construct a conversational responder using pattern matching alone.
5. Explain what your program does and does not represent.

## 2. Background — the idea in plain words

**A computer can work with symbols without knowing what they mean.**

Suppose a friend says *"I am worried about my exams."* You could reply
*"How long have you been worried about your exams?"* — and you could produce that reply
**without understanding a single word**, just by following a recipe:

1. Notice the sentence starts with **"I am"**.
2. Copy whatever comes after it: *"worried about my exams"*.
3. Change *my* → *your*.
4. Paste it into the template *"How long have you been ___?"*

That recipe is called **pattern matching**. In this lab we write it as code:

| Word | What it means here | Example |
|---|---|---|
| **token** | one word of the sentence | `'worried'` |
| **pattern** | a template with a gap in it | `['i', 'am', '?X']` |
| **variable** | the gap, written with a `?` in front | `'?X'` |
| **match** | finding which words fill the gap | `?X` = `['worried', 'about', 'my', 'exams']` |
| **binding** | the result of a match, stored in a dictionary | `{'?X': ['worried', 'about', 'my', 'exams']}` |
| **transformation** | rewriting what was captured into something new | *my exams* → *your exams* |

A famous 1960s program called **ELIZA** (Lecture 19) used exactly this trick to hold
conversations. Many people who used it felt understood — even though, as you will see
today, there is nothing in the program that understands anything.

The same "match a pattern, then rewrite" idea also works on **maths**: the rule
*"anything × 1 = that thing"* lets a program simplify `1 * x + 0` to `x` without knowing
any algebra. You will build that in Task 3.

**Key points to keep in mind while you work:**

- A pattern with a variable matches many inputs; the match returns what the variable captured.
- The program has no idea what any word refers to — it only looks at the *shape* (which word is where).
- Swapping person words (*my* → *your*) is what makes a reply sound as if it listened.
- If a fluent reply convinces a person, that tells us something about **people**, not about the program.

## 3. Setup

**Software:** Python 3.10 or later. No packages to install.

Run the cell below **first, every time you open the notebook.** It creates the blank
`___` and a small `check()` helper that the CHECK cells use. You do not need to understand
how `check()` works.

In [1]:
___ = None          # the blank you will replace with your answers

def check(label, got, expected, hint=''):
    """Print OK if got == expected, otherwise NOT YET with a hint."""
    if got == expected:
        print('OK      ', label)
    else:
        print('NOT YET ', label)
        print('          expected:', repr(expected))
        print('          you gave:', repr(got))
        if hint:
            print('          hint    :', hint)

print('Setup done. You are ready to start.')

Setup done. You are ready to start.


## 4. Python warm-up — only what you need today

Five small Python ideas are used in this lab. Each has a demo cell (just run it) and a
tiny exercise. If you already know them, run the cells quickly and move on.

### Warm-up 1 — Splitting a string into a list of words

A **string** is text in quotes. `.split()` cuts it at the spaces and gives a **list**.
`.lower()` makes everything lowercase, so `"I"` and `"i"` are treated the same.

In [2]:
text = 'I Need More Time'
print(text.lower())            # all lowercase
print(text.split())            # a list of words
print(text.lower().split())    # both: lowercase, then split

i need more time
['I', 'Need', 'More', 'Time']
['i', 'need', 'more', 'time']


**Your turn:** make `words` the lowercase list of words in `msg`.

In [3]:
msg = 'My Laptop Is Slow'
words = msg.lower().split()
print(words)

['my', 'laptop', 'is', 'slow']


In [4]:
# CHECK warm-up 1
check('lowercase word list', words, ['my', 'laptop', 'is', 'slow'],
      'use msg.lower().split()')

OK       lowercase word list


### Warm-up 2 — Indexing and slicing a list

Positions in a list start at **0**. `words[0]` is the first word.
A **slice** `words[2:]` means "from position 2 to the end", and `words[1:3]` means
"positions 1 and 2" (it **stops before** 3).

| position | 0 | 1 | 2 | 3 | 4 |
|---|---|---|---|---|---|
| word | `'i'` | `'am'` | `'worried'` | `'about'` | `'exams'` |

In [5]:
words = ['i', 'am', 'worried', 'about', 'exams']
print(words[0])      # first word
print(words[2])      # third word
print(words[2:])     # from position 2 to the end
print(words[1:3])    # positions 1 and 2
print(len(words))    # how many words

i
worried
['worried', 'about', 'exams']
['am', 'worried']
5


**Your turn:** get the words *after* `'i', 'am'` — that is, from position 2 to the end.

In [6]:
after_i_am = words[2:]
print(after_i_am)

['worried', 'about', 'exams']


In [7]:
# CHECK warm-up 2
check('slice from position 2', after_i_am, ['worried', 'about', 'exams'],
      'a slice that starts at 2 and runs to the end is words[2:]')

OK       slice from position 2


### Warm-up 3 — Dictionaries and `.get()`

A **dictionary** stores pairs: a **key** and its **value**. We use one to swap words.
`d.get(key, default)` looks the key up, and if it is not there, gives back the default
instead of crashing.

In [8]:
swap = {'my': 'your', 'i': 'you'}
print(swap['my'])                 # 'your'
print(swap.get('my', 'my'))       # found     -> 'your'
print(swap.get('exams', 'exams')) # not found -> the default, 'exams'

your
your
exams


**Your turn:** look up `'i'` in `swap`, with `'i'` as the default.

In [9]:
result = swap.get('i', 'i')
print(result)

you


In [10]:
# CHECK warm-up 3
check("swap.get('i', 'i')", result, 'you', "write swap.get('i', 'i')")

OK       swap.get('i', 'i')


### Warm-up 4 — `for` loops and `if`

A `for` loop visits each item of a list in turn. `if` runs code only when a test is true.

In [11]:
for w in ['i', 'am', 'happy']:
    if w == 'am':
        print(w, '<- this is the word am')
    else:
        print(w)

i
am <- this is the word am
happy


**Your turn:** count how many times the word `'my'` appears in `sentence_words`.

In [12]:
sentence_words = ['my', 'friend', 'took', 'my', 'pen']
count = 0
for w in sentence_words:
    if w == 'my':
        count = count + 1
print(count)

2


In [13]:
# CHECK warm-up 4
check("count of 'my'", count, 2, "the test should be: if w == 'my':")

OK       count of 'my'


### Warm-up 5 — Functions, `return` and `None`

A **function** is a named recipe. `return` sends an answer back. Returning **`None`**
is Python's way of saying *"no answer"* — our matcher will return `None` when a pattern
does not fit.

In [14]:
def first_word(words):
    if len(words) == 0:
        return None          # nothing to give back
    return words[0]

print(first_word(['hello', 'there']))
print(first_word([]))

hello
None


**Your turn:** finish `last_word` so it returns the last word, or `None` for an empty list. (Hint: the last position is `len(words) - 1`, or simply `words[-1]`.)

In [15]:
def last_word(words):
    if len(words) == 0:
        return None
    return words[-1]

print(last_word(['i', 'need', 'sleep']))

sleep


In [16]:
# CHECK warm-up 5
check('last word', last_word(['i', 'need', 'sleep']), 'sleep', 'return words[-1]')
check('empty list', last_word([]), None, 'an empty list should give None')

OK       last word
OK       empty list


## 5. Procedure — build the machine one piece at a time

Each step below introduces **one** idea. Run the demo, read the explanation, then do the
small exercise and its CHECK.

### Step 1 — A sentence as structure, not text

As a string, `'i am worried about my exams'` is one long piece of text — you cannot easily
ask *"what is the third word?"*. As a **list of tokens** you can look at, compare and
rearrange each word. Every program in this lab works on token lists.

In [17]:
sentence = 'I am worried about my exams'
tokens = sentence.lower().split()
print('as a string :', repr(sentence))
print('as tokens   :', tokens)
print('third token :', tokens[2])

as a string : 'I am worried about my exams'
as tokens   : ['i', 'am', 'worried', 'about', 'my', 'exams']
third token : worried


**Exercise 1:** turn `s1` into tokens, then pick out the word `'time'` by its position.

In [18]:
s1 = 'I need more time'
t1 = s1.lower().split()
fourth = t1[3]
print(t1, '->', fourth)

['i', 'need', 'more', 'time'] -> time


In [19]:
# CHECK Step 1
check('tokens of s1', t1, ['i', 'need', 'more', 'time'], 'use s1.lower().split()')
check("the word 'time'", fourth, 'time', "'time' is at position 3 (counting from 0)")

OK       tokens of s1
OK       the word 'time'


### Step 2 — A matcher for a pattern with one variable at the end

The pattern `['i', 'am', '?X']` means:

> *the first word must be* `i`, *the second must be* `am`, *and* `?X` *captures everything after that.*

The function `match` walks along the pattern and the sentence **side by side**:

- If the pattern part is a **fixed word**, the sentence word in the same position must be identical — otherwise give up and return `None`.
- If the pattern part is a **variable** (starts with `?`), capture all the remaining words and stop.

Read the code slowly — the comments explain every line. Then run it.

In [20]:
def match(pattern, tokens):
    """Match a list of tokens against a pattern whose LAST part may be a variable.
    Returns a dictionary of what each variable captured, or None if no match."""
    bindings = {}                              # we will store captured words here
    for i in range(len(pattern)):              # i = 0, 1, 2, ... one per pattern part
        p = pattern[i]
        if p.startswith('?'):                  # a variable such as '?X'
            bindings[p] = tokens[i:]           #   capture every word from here on
            return bindings                    #   and we are finished
        if i >= len(tokens) or p != tokens[i]: # a fixed word that is missing or different
            return None                        #   -> the pattern does not fit
    if len(pattern) == len(tokens):            # no variable, and every word matched
        return bindings
    return None                                # sentence had extra words left over

print(match(['i', 'am', '?X'], 'i am worried about my exams'.split()))
print(match(['i', 'am', '?X'], 'she is worried'.split()))

{'?X': ['worried', 'about', 'my', 'exams']}
None


**What happened?**

| Call | Why |
|---|---|
| first | `i` = `i` ✔, `am` = `am` ✔, then `?X` captures `['worried', 'about', 'my', 'exams']` |
| second | `i` ≠ `she` ✘ → the very first fixed word fails, so `None` |

**See it work step by step.** The version below is the same matcher with `print`
statements added, so you can watch each comparison.

In [21]:
def match_verbose(pattern, tokens):
    bindings = {}
    for i in range(len(pattern)):
        p = pattern[i]
        if p.startswith('?'):
            bindings[p] = tokens[i:]
            print('   %-6s is a variable -> captures %s' % (p, tokens[i:]))
            return bindings
        word = tokens[i] if i < len(tokens) else '(nothing)'
        if i >= len(tokens) or p != tokens[i]:
            print('   %-6s vs %-10s different -> NO MATCH' % (p, word))
            return None
        print('   %-6s vs %-10s same' % (p, word))
    return bindings if len(pattern) == len(tokens) else None

print('Pattern i need ?X  on  "i need a holiday"')
print('  result:', match_verbose(['i', 'need', '?X'], 'i need a holiday'.split()))
print()
print('Pattern i need ?X  on  "i want a holiday"')
print('  result:', match_verbose(['i', 'need', '?X'], 'i want a holiday'.split()))

Pattern i need ?X  on  "i need a holiday"
   i      vs i          same
   need   vs need       same
   ?X     is a variable -> captures ['a', 'holiday']
  result: {'?X': ['a', 'holiday']}

Pattern i need ?X  on  "i want a holiday"
   i      vs i          same
   need   vs want       different -> NO MATCH
  result: None


**Exercise 2 — predict, then check.** For each call, write what you think `match`
returns: a dictionary, or `None`. Write `None` without quotes. Then run the CHECK.

In [22]:
# Predict what match(...) returns for each line, then fill in the answers.
# a) match(['i', 'need', '?X'], ['i', 'need', 'sleep'])
# b) match(['i', 'need', '?X'], ['you', 'need', 'sleep'])
# c) match(['hello'],          ['hello'])
# d) match(['i', 'am', '?X'],   ['i', 'am'])          <- careful!

answer_a = {'?X': ['sleep']}
answer_b = None
answer_c = {}
answer_d = {'?X': []}

In [23]:
# CHECK Step 2
check('a)', answer_a, match(['i', 'need', '?X'], ['i', 'need', 'sleep']),
      "the variable captures ['sleep'], so the answer is {'?X': ['sleep']}")
check('b)', answer_b, match(['i', 'need', '?X'], ['you', 'need', 'sleep']),
      "the first fixed word 'i' does not equal 'you'")
check('c)', answer_c, match(['hello'], ['hello']),
      'it matches, but there is no variable, so the dictionary is empty: {}')
check('d)', answer_d, match(['i', 'am', '?X'], ['i', 'am']),
      'nothing is left after "i am", so ?X captures an EMPTY list: {"?X": []}')

OK       a)
OK       b)
OK       c)
OK       d)


### Step 3 — Transforming what was captured: swapping person words

If the user says *"my exams"*, the reply should say *"your exams"*. A dictionary `SWAP`
says which word to change into which. `swap_person` goes through the words and replaces
each one that is in `SWAP`, leaving the others alone (that is what `.get(t, t)` does —
look `t` up, and if it is not there, keep `t`).

In [24]:
SWAP = {'my': 'your', 'i': 'you', 'me': 'you', 'am': 'are', 'mine': 'yours'}

def swap_person(tokens):
    new_words = []
    for t in tokens:
        new_words.append(SWAP.get(t, t))   # swapped word, or the same word
    return new_words

captured = 'worried about my exams'.split()
print('captured:', captured)
print('swapped :', swap_person(captured))

captured: ['worried', 'about', 'my', 'exams']
swapped : ['worried', 'about', 'your', 'exams']


**Exercise 3:** the sentence *"I am angry with myself because I was late"* does not swap
properly yet — `myself` and `was` stay the same. Add two pairs to `SWAP`:
`'myself'` → `'yourself'` and `'was'` → `'were'`.

In [25]:
SWAP['myself'] = 'yourself'
SWAP['was'] = 'were'

print(swap_person('angry with myself because i was late'.split()))

['angry', 'with', 'yourself', 'because', 'you', 'were', 'late']


In [26]:
# CHECK Step 3
check('swap a full phrase',
      swap_person('angry with myself because i was late'.split()),
      ['angry', 'with', 'yourself', 'because', 'you', 'were', 'late'],
      "SWAP['myself'] = 'yourself' and SWAP['was'] = 'were'")

OK       swap a full phrase


### Step 4 — Putting it together: a responder

A **rule** is a pair: a *pattern* and a *reply template*. In the template, `{X}` marks
where the captured (and swapped) words go.

The responder:

1. Turns the sentence into tokens (lowercase, without a final `.`, `!` or `?`).
2. Tries each rule **in order**. The first rule whose pattern matches wins.
3. Swaps person words in what was captured and pastes them into the template.
4. If **no** rule matches, gives a general **fallback** reply.

In [27]:
RULES = [
    (['i', 'am', '?X'],      'How long have you been {X}?'),
    (['i', 'need', '?X'],    'Why do you need {X}?'),
    (['i', 'cannot', '?X'],  'What stops you being able to {X}?'),
]
FALLBACK = 'Tell me more about that.'

def respond(sentence, rules=RULES):
    tokens = sentence.lower().strip('.!?').split()      # 1. tokens
    for pattern, template in rules:                     # 2. try each rule
        b = match(pattern, tokens)
        if b is not None:                               #    this rule fits
            captured = swap_person(b.get('?X', []))     # 3. swap person words
            return template.format(X=' '.join(captured))
    return FALLBACK                                     # 4. nothing fitted

for s in ['I am worried about my exams.',
          'I need more time',
          'I cannot sleep at night',
          'The weather is fine today']:
    print('%-32s -> %s' % (s, respond(s)))

I am worried about my exams.     -> How long have you been worried about your exams?
I need more time                 -> Why do you need more time?
I cannot sleep at night          -> What stops you being able to sleep at night?
The weather is fine today        -> Tell me more about that.


**Exercise 4:** what reply does *"I am tired of my phone"* get? Write the exact reply
you expect as a string, then run the CHECK.

In [28]:
predicted_reply = 'How long have you been tired of your phone?'

In [29]:
# CHECK Step 4
check('predicted reply', predicted_reply, respond('I am tired of my phone'),
      "rule 1 fits; ?X = 'tired of my phone', then my -> your")

OK       predicted reply


## 6. Tasks (marked)

These are the same five tasks as Practical 02, worth the same marks. Each one is split
into small parts, with blanks to fill in and a CHECK at the end.

### Task 1 — Add five rules of your own

> **Marks:** 4

Add **five** new rules to `MY_RULES`. At least **two** of them must capture a phrase with
`?X` and reuse it in the reply with `{X}`.

Two examples are already filled in (and do count as two of your five). Fill in the three
blank rules — replace each `___`. A pattern is a **list of words in quotes**, and a reply
is **one string in quotes**.

Ideas: `['i', 'want', '?X']`, `['i', 'think', '?X']`, `['i', 'hate', '?X']`,
`['everyone', '?X']`, `['yes']`, `['hello']`.

In [30]:
MY_RULES = [
    (['i', 'feel', '?X'],   'Why do you feel {X}?'),          # example 1 - captures
    (['hello'],             'Hello! What is on your mind?'),  # example 2 - no variable
    (['i', 'want', '?X'],   'What do you want {X}?'),
    (['i', 'think', '?X'],  'What makes you think {X}?'),
    (['i', 'hate', '?X'],   'Why do you hate {X}?'),
]

# Your rules are tried FIRST; the three rules from Step 4 are used if none of yours fits.
ALL_RULES = MY_RULES + RULES

for pattern, reply in MY_RULES:
    print(pattern, '->', reply)

['i', 'feel', '?X'] -> Why do you feel {X}?
['hello'] -> Hello! What is on your mind?
['i', 'want', '?X'] -> What do you want {X}?
['i', 'think', '?X'] -> What makes you think {X}?
['i', 'hate', '?X'] -> Why do you hate {X}?


Now write **one test sentence for each** of your five rules, in the same order, and run
the responder on them.

In [31]:
MY_TESTS = [
    'I feel tired of my timetable',      # tests example 1
    'Hello',                             # tests example 2
    'I want a new laptop',
    'I think exams are difficult',
    'I hate noisy classrooms',
]

for s in MY_TESTS:
    print('%-36s -> %s' % (s, respond(s, ALL_RULES)))

I feel tired of my timetable         -> Why do you feel tired of your timetable?
Hello                                -> Hello! What is on your mind?
I want a new laptop                  -> What do you want a new laptop?
I think exams are difficult          -> What makes you think exams are difficult?
I hate noisy classrooms              -> Why do you hate noisy classrooms?


In [32]:
# CHECK Task 1
def _task1():
    if len(MY_RULES) < 5:
        print('NOT YET  you need at least 5 rules in MY_RULES')
        return
    for k, rule in enumerate(MY_RULES):
        if (not isinstance(rule, tuple) or len(rule) != 2 or not isinstance(rule[0], list)
                or not isinstance(rule[1], str)):
            print('NOT YET  rule %d must look like ([words...], "reply")  - got %r' % (k + 1, rule))
            return
    capturing = [r for r in MY_RULES if '?X' in r[0] and '{X}' in r[1]]
    check('at least 2 rules capture ?X and reuse {X}', len(capturing) >= 2, True,
          "put '?X' in the pattern AND '{X}' in the reply")
    for k, (rule, s) in enumerate(zip(MY_RULES, MY_TESTS)):
        fits = isinstance(s, str) and match(rule[0], s.lower().strip('.!?').split()) is not None
        check('test %d matches rule %d' % (k + 1, k + 1), fits, True,
              'test sentence %r should start with the fixed words of %r' % (s, rule[0]))
_task1()

OK       at least 2 rules capture ?X and reuse {X}
OK       test 1 matches rule 1
OK       test 2 matches rule 2
OK       test 3 matches rule 3
OK       test 4 matches rule 4
OK       test 5 matches rule 5


### Task 2 — A variable in the middle of a pattern

> **Marks:** 5

Our `match` can only capture at the **end** of a pattern. Look what happens with a
variable in the middle:

In [33]:
print(match(['i', '?X', 'about', 'exams'], 'i am worried about exams'.split()))

{'?X': ['am', 'worried', 'about', 'exams']}


It captured `am worried about exams` — it grabbed everything and **never checked** the
words `about` and `exams` that come after the variable. That is wrong.

**The idea for the fix.** The fixed word right after the variable is called the
**anchor**. To find out what `?X` captures:

1. Look up the **anchor** (here `'about'`) in the rest of the sentence.
2. `?X` captures the words **from where we are, up to (not including) the anchor**.
3. Carry on matching from the anchor.

```
pattern :  i    ?X              about   exams
sentence:  i    am    worried   about   exams
position:  0    1     2         3       4
                └──── ?X ────┘  ↑
                                anchor found at position 3, so ?X = tokens[1:3]
```

Because the variable no longer uses up "everything", we now need a second counter:
`j` for our place in the **pattern**, and `i` for our place in the **sentence**.

Useful Python: `tokens.index('about', i)` gives the position of `'about'`, searching from
position `i` onwards.

Fill in the **four** blanks in `match2`.

In [34]:
def match2(pattern, tokens):
    """Like match, but a variable may sit anywhere - also between two fixed words."""
    bindings = {}
    i = 0                                         # our position in the SENTENCE
    for j in range(len(pattern)):                 # j = our position in the PATTERN
        p = pattern[j]
        if p.startswith('?'):                     # ---- a variable ----
            if j == len(pattern) - 1:             # last part of the pattern:
                bindings[p] = tokens[i:]          # blank 1: capture everything left
                return bindings
            anchor = pattern[j + 1]               # the fixed word after the variable
            if anchor not in tokens[i:]:
                return None                       # anchor is missing -> no match
            end = tokens.index(anchor, i)         # where the anchor sits
            bindings[p] = tokens[i:end]           # blank 2: words from i up to the anchor
            i = end                               # carry on from the anchor
        else:                                     # ---- a fixed word ----
            if i >= len(tokens) or tokens[i] != p:
                return None                       # blank 3: what do we return on a mismatch?
            i = i + 1                             # blank 4: move one word along the sentence
    if i != len(tokens):                          # words left over -> not a full match
        return None
    return bindings

Run these tests. Predict each result first.

In [35]:
tests2 = [
    (['i', '?X', 'about', 'exams'],  'i am worried about exams'),       # middle variable
    (['?A', 'is', '?B'],             'the deadline is next monday'),    # two variables
    (['i', 'am', '?X'],              'i am worried about my exams'),    # end variable still works
    (['i', 'am', '?X'],              'she is worried'),                 # should be None
    (['i', '?X', 'about', 'exams'],  'i am worried about my marks'),    # anchor 'exams' missing
]
for pattern, sentence in tests2:
    print('%-28s %-30s -> %s' % (' '.join(pattern), sentence, match2(pattern, sentence.split())))

i ?X about exams             i am worried about exams       -> {'?X': ['am', 'worried']}
?A is ?B                     the deadline is next monday    -> {'?A': ['the', 'deadline'], '?B': ['next', 'monday']}
i am ?X                      i am worried about my exams    -> {'?X': ['worried', 'about', 'my', 'exams']}
i am ?X                      she is worried                 -> None
i ?X about exams             i am worried about my marks    -> None


In [36]:
# CHECK Task 2
try:
    check('middle variable', match2(['i', '?X', 'about', 'exams'], 'i am worried about exams'.split()),
          {'?X': ['am', 'worried']}, 'blank 2 is the slice tokens[i:end]')
    check('two variables', match2(['?A', 'is', '?B'], 'the deadline is next monday'.split()),
          {'?A': ['the', 'deadline'], '?B': ['next', 'monday']}, 'blanks 1 and 2 both capture')
    check('variable at the end', match2(['i', 'am', '?X'], 'i am worried'.split()),
          {'?X': ['worried']}, 'blank 1 is tokens[i:] and blank 4 is i + 1')
    check('no match', match2(['i', 'am', '?X'], 'she is worried'.split()),
          None, 'blank 3: a mismatch returns None')
    check('extra words', match2(['i', '?X', 'about', 'exams'], 'i am worried about exams today'.split()),
          None, "the pattern ends at 'exams' but the sentence has 'today' left over")
except Exception as err:
    print('NOT YET  match2 raised an error:', type(err).__name__, '-', err)
    print('         (usually a blank that still says ___)')

OK       middle variable
OK       two variables
OK       variable at the end
OK       no match
OK       extra words


**Use it in the responder.** With `match2`, rules can now have the variable in the middle.
Run this cell — the responder below is the same as Step 4, but calls `match2`.

In [37]:
def respond2(sentence, rules):
    tokens = sentence.lower().strip('.!?').split()
    for pattern, template in rules:
        b = match2(pattern, tokens)
        if b is not None:
            captured = swap_person(b.get('?X', []))
            return template.format(X=' '.join(captured))
    return FALLBACK

MIDDLE_RULES = [(['i', '?X', 'my', 'teacher'], 'Why do you {X} your teacher?')] + ALL_RULES
print(respond2('I really like my teacher', MIDDLE_RULES))
print(respond2('I am worried about my exams', MIDDLE_RULES))

Why do you really like your teacher?
How long have you been worried about your exams?


### Task 3 — A simple expression simplifier

> **Marks:** 5

The same match-and-rewrite idea works on maths. We want a program that turns
`1 * x + 0` into just `x` — using **rewrite rules**, not algebra knowledge.

**Part A — writing an expression as a nested list.**
We write every operation as a list of three things: `[operator, left, right]`.

| Maths | Nested list |
|---|---|
| `1 * x` | `['*', 1, 'x']` |
| `x + 0` | `['+', 'x', 0]` |
| `1 * x + 0` | `['+', ['*', 1, 'x'], 0]` ← the `1 * x` list sits **inside** the `+` list |

Think of it as boxes inside boxes. Run the cell and see how to reach inside.

In [38]:
expr = ['+', ['*', 1, 'x'], 0]        # means (1 * x) + 0

print('whole       :', expr)
print('operator    :', expr[0])        # '+'
print('left part   :', expr[1])        # ['*', 1, 'x']  - itself a list
print('right part  :', expr[2])        # 0
print('inside left :', expr[1][2])     # 'x'

whole       : ['+', ['*', 1, 'x'], 0]
operator    : +
left part   : ['*', 1, 'x']
right part  : 0
inside left : x


**Exercise A:** write `x * 1 + 0` and `y + 0` as nested lists.

In [39]:
e1 = ['+', ['*', 'x', 1], 0]       # x * 1 + 0
e2 = ['+', 'y', 0]                 # y + 0

In [40]:
# CHECK Task 3 part A
check('x * 1 + 0', e1, ['+', ['*', 'x', 1], 0], "the outer list is the '+'; its left part is ['*', 'x', 1]")
check('y + 0', e2, ['+', 'y', 0], "['+', 'y', 0]")

OK       x * 1 + 0
OK       y + 0


**A helper to print expressions the normal way.** You do not need to change this; it
turns `['+', ['*', 1, 'x'], 0]` back into `((1 * x) + 0)` so the output is easy to read.

In [41]:
def show(e):
    if not isinstance(e, list):           # a plain number or letter
        return str(e)
    return '(' + show(e[1]) + ' ' + e[0] + ' ' + show(e[2]) + ')'

print(show(['+', ['*', 1, 'x'], 0]))

((1 * x) + 0)


**Part B — the rewrite rules.** `simplify_node` looks at **one** `[op, left, right]` list
and applies the first rule that fits. The first rule is done for you; fill in the rest.

| Rule | Meaning |
|---|---|
| `1 * x → x` | multiplying by 1 changes nothing *(done for you)* |
| `x * 1 → x` | same, with the 1 on the right |
| `x + 0 → x` | adding 0 changes nothing |
| `0 + x → x` | same, with the 0 on the left |
| `x * 0 → 0` | anything times 0 is 0 |

In [42]:
def simplify_node(e):
    op, left, right = e[0], e[1], e[2]
    if op == '*' and left == 1:
        return right                              # 1 * x -> x   (done for you)
    if op == '*' and right == 1:
        return left                               # x * 1 -> x
    if op == '+' and right == 0:
        return left                               # x + 0 -> x
    if op == '+' and left == 0:
        return right                              # 0 + x -> x
    if op == '*' and (left == 0 or right == 0):
        return 0                                  # x * 0 -> 0
    return e                                      # no rule fits: leave it alone

In [43]:
# CHECK Task 3 part B
check('1 * x', simplify_node(['*', 1, 'x']), 'x')
check('x * 1', simplify_node(['*', 'x', 1]), 'x', 'return left')
check('x + 0', simplify_node(['+', 'x', 0]), 'x', 'return left')
check('0 + x', simplify_node(['+', 0, 'x']), 'x', 'return right')
check('x * 0', simplify_node(['*', 'x', 0]), 0, 'return 0 (the number, no quotes)')
check('no rule', simplify_node(['+', 'x', 'y']), ['+', 'x', 'y'], 'should come back unchanged')

OK       1 * x
OK       x * 1
OK       x + 0
OK       0 + x
OK       x * 0
OK       no rule


**Part C — simplifying the insides first (given).** `simplify_node` only looks at the
outer box. In `['+', ['*', 1, 'x'], 0]` the `1 * x` is *inside*. So `simplify_once` first
simplifies the left and right parts, and **then** the outer box. (A function that calls
itself like this is called **recursive** — just think "open the inner boxes first".)

Run it and watch the trace.

In [44]:
def simplify_once(e):
    if not isinstance(e, list):                   # a number or letter: nothing to do
        return e
    left = simplify_once(e[1])                    # simplify the inner boxes first
    right = simplify_once(e[2])
    before = [e[0], left, right]
    after = simplify_node(before)                 # then the outer box
    if after != before:
        print('     rule applied:', show(before), '->', show(after))
    return after

print(show(simplify_once(['+', ['*', 1, 'x'], 0])))

     rule applied: (1 * x) -> x
     rule applied: (x + 0) -> x
x


**Part D — repeat until nothing changes.** A rewrite system keeps applying its rules
**until a whole pass changes nothing**. Fill in the two blanks: stop when `new` is the
same as `e`; otherwise carry on with `new`.

In [45]:
def simplify(e):
    print('start :', show(e))
    pass_number = 1
    while True:
        print(' pass', pass_number)
        new = simplify_once(e)
        if new == e:                              # blank 1: nothing changed?
            print('result:', show(new))
            return new
        e = new                                   # blank 2: keep the simplified version
        pass_number = pass_number + 1

In [46]:
simplify(['+', ['*', 1, 'x'], 0])                  # 1 * x + 0
print()
simplify(['*', ['+', 'y', 0], ['*', 1, 1]])        # (y + 0) * (1 * 1)
print()
simplify(['+', ['*', 'x', 0], 'z'])                # x * 0 + z

start : ((1 * x) + 0)
 pass 1
     rule applied: (1 * x) -> x
     rule applied: (x + 0) -> x
 pass 2
result: x

start : ((y + 0) * (1 * 1))
 pass 1
     rule applied: (y + 0) -> y
     rule applied: (1 * 1) -> 1
     rule applied: (y * 1) -> y
 pass 2
result: y

start : ((x * 0) + z)
 pass 1
     rule applied: (x * 0) -> 0
     rule applied: (0 + z) -> z
 pass 2
result: z


'z'

In [47]:
# CHECK Task 3 parts C and D
import io, contextlib
try:
    with contextlib.redirect_stdout(io.StringIO()):   # hide the trace while checking
        r1 = simplify(['+', ['*', 1, 'x'], 0])
        r2 = simplify(['*', ['+', 'y', 0], ['*', 1, 1]])
        r3 = simplify(['+', ['*', 'x', 0], 'z'])
    check('1 * x + 0', r1, 'x')
    check('(y + 0) * (1 * 1)', r2, 'y')
    check('x * 0 + z', r3, 'z')
except Exception as err:
    print('NOT YET  simplify raised an error:', type(err).__name__, '-', err)
    print('         (blank 1 should be e and blank 2 should be new)')

OK       1 * x + 0
OK       (y + 0) * (1 * 1)
OK       x * 0 + z


### Task 4 — Test it on a classmate

> **Marks:** 2

1. Ask a classmate to type 4–5 sentences into the chat cell below. **Do not** tell them
   how it works.
2. Write down their sentences and the replies in the Observations table (Section 7).
3. Afterwards, ask them: *"What do you think the program knew or understood?"* Write their
   answer down **before** you show them the rules.

Run the cell to test the responder. In this completed file, a reproducible sample session is used so the notebook runs non-interactively; replace the sample sentences with a real classmate's sentences if required by your instructor.

In [48]:
# Task 4 - reproducible classmate-session example.
# In an actual lab session, replace these with the classmate's real sentences and record the real reaction.
task4_sentences = [
    'I feel tired of my timetable',
    'Hello',
    'I want more time',
    'The weather is fine today',
    'I am worried about my exams',
]
for said in task4_sentences:
    print('You:', said)
    print('Program:', respond2(said, MIDDLE_RULES))


You: I feel tired of my timetable
Program: Why do you feel tired of your timetable?
You: Hello
Program: Hello! What is on your mind?
You: I want more time
Program: What do you want more time?
You: The weather is fine today
Program: Tell me more about that.
You: I am worried about my exams
Program: How long have you been worried about your exams?


### Task 5 — State what your program represents

> **Marks:** 4

First run this cell. It prints **everything** your program stores — its whole memory.

In [49]:
print('1. Rules  :', len(ALL_RULES), '(pattern, reply) pairs')
for pattern, reply in ALL_RULES:
    print('     ', pattern, '->', reply)
print('2. SWAP   :', SWAP)
print('3. FALLBACK:', repr(FALLBACK))
print()
print('Ask the same thing twice:')
print('  ', respond2('I am worried about my exams', MIDDLE_RULES))
print('  ', respond2('I am worried about my exams', MIDDLE_RULES))

1. Rules  : 8 (pattern, reply) pairs
      ['i', 'feel', '?X'] -> Why do you feel {X}?
      ['hello'] -> Hello! What is on your mind?
      ['i', 'want', '?X'] -> What do you want {X}?
      ['i', 'think', '?X'] -> What makes you think {X}?
      ['i', 'hate', '?X'] -> Why do you hate {X}?
      ['i', 'am', '?X'] -> How long have you been {X}?
      ['i', 'need', '?X'] -> Why do you need {X}?
      ['i', 'cannot', '?X'] -> What stops you being able to {X}?
2. SWAP   : {'my': 'your', 'i': 'you', 'me': 'you', 'am': 'are', 'mine': 'yours', 'myself': 'yourself', 'was': 'were'}
3. FALLBACK: 'Tell me more about that.'

Ask the same thing twice:
   How long have you been worried about your exams?
   How long have you been worried about your exams?


**Now complete the sentences below**

**(a) What my program actually stores** — list each thing printed above:

1. A list of pattern/reply rules in `ALL_RULES`.
2. The word-swapping dictionary `SWAP`.
3. The fallback reply string `FALLBACK`.

**(b) What a user typically assumes it stores** — think about your Task 4 tester:

1. The meaning or topic of what they said.
2. Personal facts or experiences about them.
3. A memory of previous conversations and understanding of context.

**(c) The gap:** the program does not store **the user's meaning, personal experiences, or conversational memory**,
yet users believe it does because **the fixed rules, captured phrases, and word swapping can produce replies that sound relevant**.

**(d) Evidence from the "ask the same thing twice" output:** **the same input produces the same reply both times; no new information is stored between the two calls.**


## 7. Observations

These tables use the reproducible Task 1/Task 4 examples in this completed notebook.

**Responder results (use your Task 1 tests and your Task 4 session)**

| Input sentence | Rule that matched (or "fallback") | Reply produced | Sensible? (yes / partly / no) |
|---|---|---|---|
| I feel tired of my timetable | Rule 1 (`i feel ?X`) | Why do you feel tired of your timetable? | yes |
| Hello | Rule 2 (`hello`) | Hello! What is on your mind? | yes |
| I want more time | Rule 3 (`i want ?X`) | What do you want more time? | partly |
| I think exams are difficult | Rule 4 (`i think ?X`) | What makes you think exams are difficult? | yes |
| I am worried about my exams | Step 4 rule (`i am ?X`) | How long have you been worried about your exams? | yes |

**Simplifier results (Task 3)**

| Expression | Rules applied | Result | Passes needed |
|---|---|---|---|
| `1 * x + 0` | `1 * x -> x`, then `x + 0 -> x` | `x` | 2 |
| `(y + 0) * (1 * 1)` | `y + 0 -> y`, `1 * 1 -> 1`, then `y * 1 -> y` | `y` | 2 |
| `x * 0 + z` | `x * 0 -> 0`, then `0 + z -> z` | `z` | 2 |

**Tester's reaction (Task 4)**

| What they assumed the program knew | What it actually stores |
|---|---|
| It understood the topic of the sentence. | It stores patterns, reply templates, and captured words. |
| It remembered something about the speaker. | It has no personal memory of the speaker; the same input is handled by the same rules. |


## 8. Discussion

1. **Your responder produced sensible replies. What, if anything, does it understand?**
   My responder does not represent understanding of the user's meaning. It stores fixed patterns, reply templates, and a small word-swapping dictionary; it matches the shape of the input and rewrites captured words.

2. **Why does swapping person words make such a large difference to how the reply reads?**
   Without the swap, a reply to "I am worried about my exams" could repeat "my exams", which sounds as if the program is speaking from the user's point of view. Changing `my` to `your` makes the template fit the conversational perspective.

3. **Lecture 19 noted that people who knew how such a program worked still felt understood by it. What does that tell you about evaluating conversational systems today?**
   A fluent and relevant-looking reply can create an impression of understanding even when the underlying program is only matching patterns. When evaluating a chatbot, we should examine both the quality of its responses and what information or representation the system actually uses to produce them.


## 9. Viva questions

Be ready to answer these on the spot. They test whether you understand what you ran, not
whether you can recite the code.

1. Show me where in your code the program decides what a sentence means. (There is no such place — can you say why?)
2. What does your matcher return when the pattern fits, and what when it does not?
3. Why did we represent the sentence as tokens rather than as a string?
4. What happens to your responder when no rule matches, and is that a good design?
5. How is your matcher related to the unification you will write next week?
6. Your tester assumed the program knew something. What, and why did they assume it?

## 10. Submission checklist

- [ ] Notebook runs top to bottom from a clean kernel (Kernel → Restart & Run All)
- [ ] Every CHECK cell prints `OK`
- [ ] Task 1 has five rules of your own and a test sentence for each
- [ ] The observations tables are filled in with your actual results
- [ ] Task 5 and the discussion answers are written in your own words
- [ ] Your name and roll number are in the cell below
- [ ] File named `P02a_<your-roll-number>.ipynb`

In [50]:
NAME = "Muhammad Shehmeer"
ROLL_NUMBER = "2025-cse-001"
print(NAME, ROLL_NUMBER)

Muhammad Shehmeer 2025-cse-001
